# 🎵 Music Genre Recognition using AI/ML

This notebook implements a CNN-based Music Genre Recognition system using the GTZAN dataset.

## 1. Install and Import Libraries

In [1]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [ ]:

!pip install librosa tensorflow numpy matplotlib scikit-learn


## 2. Import Required Libraries

In [3]:

import librosa
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


## 3. Feature Extraction - Mel Spectrogram

In [5]:
MAX_LEN = 1300

def extract_mel_spectrogram(file_path):
    try:
        audio, sr = librosa.load(file_path, duration=30, mono=True)

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=sr,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Normalize to [0, 1]
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

        # Pad / truncate
        if mel_db.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - mel_db.shape[1]
            mel_db = np.pad(mel_db, ((0, 0), (0, pad_width)))
        else:
            mel_db = mel_db[:, :MAX_LEN]

        return mel_db

    except Exception as e:
        return None


## 4. Load Dataset

In [ ]:
import os
dataset_path ="GTZAN"
print(os.listdir(dataset_path))

In [17]:

dataset_path ="GTZAN"
genres = os.listdir(dataset_path)

X, y = [], []

for label, genre in enumerate(genres):
    genre_path = os.path.join(dataset_path, genre)
    for file in os.listdir(genre_path):
        mel = extract_mel_spectrogram(os.path.join(genre_path, file))
        if mel is not None:
            X.append(mel)
            y.append(label)
X = np.array(X)
y = np.array(y)
X = X[..., np.newaxis]


C:\Users\nmtha\AppData\Local\Temp\ipykernel_85584\3004666631.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, duration=30, mono=True)


## 5. Train-Test Split

In [19]:
print(X.shape)

(999, 128, 1300, 1)


In [15]:
X = X.squeeze(-1)

ValueError: cannot select an axis to squeeze out which has size not equal to one

In [58]:
import numpy as np

X = np.array(X)
y = np.array(y)
X = X[..., np.newaxis]

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [23]:
print(X_train.shape)


(799, 128, 1300, 1)


## 6. CNN Model

In [25]:
model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=X_train.shape[1:]),
    MaxPooling2D((2,2)),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.summary()


G:\Anaconda\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 1298, 16)       │             160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 649, 16)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 647, 32)         │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 323, 32)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 321, 64)         │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 160, 64)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 143360)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │      18,350,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 18,374,794 (70.09 MB)

 Trainable params: 18,374,794 (70.09 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Train Model

In [27]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [29]:

history = model.fit(
    X_train, y_train,
    epochs=28,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 18s 653ms/step - accuracy: 0.1414 - loss: 2.7016 - val_accuracy: 0.1500 - val_loss: 2.2335
Epoch 2/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 641ms/step - accuracy: 0.1902 - loss: 2.0952 - val_accuracy: 0.1950 - val_loss: 2.0325
Epoch 3/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 643ms/step - accuracy: 0.3066 - loss: 1.8743 - val_accuracy: 0.3550 - val_loss: 1.8212
Epoch 4/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 16s 656ms/step - accuracy: 0.4330 - loss: 1.6030 - val_accuracy: 0.4500 - val_loss: 1.5869
Epoch 5/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 665ms/step - accuracy: 0.5394 - loss: 1.3357 - val_accuracy: 0.4650 - val_loss: 1.5277
Epoch 6/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 677ms/step - accuracy: 0.6033 - loss: 1.1546 - val_accuracy: 0.4900 - val_loss: 1.5216
Epoch 7/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 685ms/step - accuracy: 0.6546 - loss: 1.0162 - val_accuracy: 0.5600 - val_loss: 1.2557
Epoch 8/28
25/25 ━━━━━━━━━━━━━━━━━━━━ 17s 678ms/step - accuracy: 0.7597 - loss: 0.7218 - val_accu

## 8. Evaluation

In [31]:

test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

y_pred = np.argmax(model.predict(X_test), axis=1)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.6150 - loss: 1.7595
Test Accuracy: 0.6150000095367432
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step
[[12  0  3  0  0  2  2  0  0  2]
 [ 0 11  0  0  0  1  0  0  0  0]
 [ 0  0 17  0  0  1  0  1  1  4]
 [ 2  0  1  9  4  0  0  3  3  0]
 [ 1  0  0  0 10  0  0  0  2  2]
 [ 2  3  1  0  0 21  0  0  0  0]
 [ 0  0  0  0  0  0 17  0  0  1]
 [ 0  0  1  3  1  1  0 10  2  1]
 [ 1  0  1  4  2  1  0  1 12  0]
 [ 3  0  4  0  1  3  1  2  2  4]]
              precision    recall  f1-score   support

           0       0.57      0.57      0.57        21
           1       0.79      0.92      0.85        12
           2       0.61      0.71      0.65        24
           3       0.56      0.41      0.47        22
           4       0.56      0.67      0.61        15
           5       0.70      0.78      0.74        27
           6       0.85      0.94      0.89        18
           7       0.59      0.53      0.56        19
           8       0.55      0.55   

## 

## 9. Predict Genre for New Song

In [33]:

def predict_genre(file_path):
    mel = extract_mel_spectrogram(file_path)
    mel = mel[np.newaxis, ..., np.newaxis]
    prediction = model.predict(mel)
    return genres[np.argmax(prediction)]

print(predict_genre("test.wav"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
pop


In [37]:
model.save("genre_cnn_model.h5")